In [ ]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 140.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [ ]:
!nvidia-smi

Wed May 20 08:29:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="GkpOAlaNVZSYkMMNEdZf") # (Usa tu clave real aquí, claro)
project = rf.workspace("adris-workspace-y2ppn").project("black_box")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
# Cambia 'nombre_de_tu_archivo.zip' por el nombre real del archivo que acabas de subir
!unzip black_box.v1i.yolov8.zip -d mis_fotos_caja

Archive:  black_box.v1i.yolov8.zip
 extracting: mis_fotos_caja/train/images/caja_19_jpg.rf.f7ba160e76a382f4dc14eec59383e6ea.jpg  
 extracting: mis_fotos_caja/train/images/caja_74_jpg.rf.90c3c55fa30dfb2d032d480d95592b40.jpg  
 extracting: mis_fotos_caja/train/images/caja_62_jpg.rf.a4fd8d7120ce2f1f1d6b171f2751bba6.jpg  
 extracting: mis_fotos_caja/train/images/caja_72_jpg.rf.6930c976b74b8c5b5d0d7ca929371349.jpg  
 extracting: mis_fotos_caja/train/images/caja_64_jpg.rf.d78a8af3fa2d8089a942572b5ee96750.jpg  
 extracting: mis_fotos_caja/train/images/caja_52_jpg.rf.9ac36d7b527fbc61c0ef8d641ddaa2f7.jpg  
 extracting: mis_fotos_caja/train/images/caja_83_jpg.rf.76d91054015544711e1b38bc6b2221c6.jpg  
 extracting: mis_fotos_caja/train/images/caja_41_jpg.rf.385ce62c46578bf29be24760c31fc94c.jpg  
 extracting: mis_fotos_caja/train/images/caja_68_jpg.rf.ec0ed155a22c8d8133638eefbdec3204.jpg  
 extracting: mis_fotos_caja/train/images/caja_70_jpg.rf.4f36cbdb2100c3ff71807147aeeec0bd.jpg  
 extracting: mi

In [ ]:
import os
import glob

print("🔍 Buscando las 2 fotos corruptas...")

carpeta_etiquetas = '/content/mis_fotos_caja/train/labels'
carpeta_imagenes = '/content/mis_fotos_caja/train/images'
cache_file = '/content/mis_fotos_caja/train/labels.cache'

borrados = 0

# 1. Borrar la memoria caché para que la IA no recuerde el error
if os.path.exists(cache_file):
    os.remove(cache_file)
    print("🗑️ Memoria caché de YOLO borrada.")

# 2. Revisar etiqueta por etiqueta
for txt_file in glob.glob(os.path.join(carpeta_etiquetas, '*.txt')):
    with open(txt_file, 'r') as f:
        lineas = f.readlines()

    archivo_corrupto = False

    # Si está vacío o si tiene exactamente 5 datos (eso es una caja normal, un polígono tiene más de 6)
    if len(lineas) == 0:
        archivo_corrupto = True

    for linea in lineas:
        if len(linea.strip().split()) == 5:
            archivo_corrupto = True
            break

    # 3. Borrar el archivo de texto y su foto correspondiente
    if archivo_corrupto:
        os.remove(txt_file)
        # Buscar la foto (puede ser .jpg o .png)
        imagen_jpg = txt_file.replace('labels', 'images').replace('.txt', '.jpg')
        imagen_png = txt_file.replace('labels', 'images').replace('.txt', '.png')

        if os.path.exists(imagen_jpg): os.remove(imagen_jpg)
        if os.path.exists(imagen_png): os.remove(imagen_png)

        borrados += 1

print(f"✅ ¡Limpieza completada! Se han eliminado {borrados} fotos conflictivas.")

🔍 Buscando las 2 fotos corruptas...
🗑️ Memoria caché de YOLO borrada.
✅ ¡Limpieza completada! Se han eliminado 2 fotos conflictivas.


In [ ]:
import os
import yaml
from ultralytics import YOLO

def entrenar_modelo_definitivo():
    ruta_yaml = '/content/mis_fotos_caja/data.yaml'

    if not os.path.exists(ruta_yaml):
        print("❌ ERROR: No encuentro el data.yaml. ¿Descomprimiste bien el ZIP en dataset_v2?")
        return

    # 1. Aplicar el parche para la validación (porque tienes 100% en Train)
    print("[INFO] Parcheando el archivo data.yaml...")
    with open(ruta_yaml, 'r') as f:
        datos = yaml.safe_load(f)

    ruta_base = os.path.dirname(ruta_yaml)
    datos['train'] = os.path.join(ruta_base, 'train', 'images')
    datos['val'] = os.path.join(ruta_base, 'train', 'images')

    with open(ruta_yaml, 'w') as f:
        yaml.dump(datos, f)

    print("✅ ¡Parche aplicado con éxito!")

    # 2. Iniciar el Entrenamiento Intensivo
    print("🚀 Iniciando entrenamiento DEFINITIVO en GPU (100 epochs)...")

    # Cargamos la estructura base de YOLOv8
    model = YOLO('yolov8n-seg.pt')

    # Entrenamos con los nuevos parámetros
    results = model.train(
        data=ruta_yaml,
        epochs=100,           # ¡El doble de tiempo de estudio!
        imgsz=640,            # El tamaño perfecto que cuadramos en Roboflow
        task='segment',       # Seguimos haciendo máscaras/recortes
        project='RobotG1',    # Carpeta principal
        name='modelo_final'   # Subcarpeta donde guardará tu best.pt
    )

    print("🎉 ¡ENTRENAMIENTO COMPLETADO! Busca tu best.pt en la carpeta RobotG1/modelo_final/weights/")

# Ejecutar la función
entrenar_modelo_definitivo()

[INFO] Parcheando el archivo data.yaml...
✅ ¡Parche aplicado con éxito!
🚀 Iniciando entrenamiento DEFINITIVO en GPU (100 epochs)...
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mis_fotos_caja/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.